# 1. 데이터 로드 및 전처리

## 1.1 필요한 라이브러리 불러오기

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
# 데이터가 저장된 기본 경로
base_path = 'datasets/val_datasets'


## 1.2 Parquet 파일 불러오기 및 전처리

In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm

def load_parquet_files(file_paths):
    """Load multiple parquet files into a single pandas DataFrame."""
    return pd.concat([pd.read_parquet(file) for file in tqdm(file_paths)])

def load_parquet_data(base_path):
    """Load all required parquet datasets from a specified base path."""
    acc_files = [
        'ch2024_val__m_acc_part_1.parquet.gzip',
        'ch2024_val__m_acc_part_2.parquet.gzip',
        'ch2024_val__m_acc_part_3.parquet.gzip',
        'ch2024_val__m_acc_part_4.parquet.gzip'
    ]
    
    acc_file_paths = [f'{base_path}/{file}' for file in acc_files]
    acc_data = load_parquet_files(acc_file_paths)

    data_files = {
        'activity_data': 'ch2024_val__m_activity.parquet.gzip',
        'hr_data': 'ch2024_val__w_heart_rate.parquet.gzip',
        'step_count': 'ch2024_val__w_pedo.parquet.gzip',
        'light': 'ch2024_val__m_light.parquet.gzip',
        'gps': 'ch2024_val__m_gps.parquet.gzip',
    }

    data = {name: pd.read_parquet(f'{base_path}/{file}') for name, file in data_files.items()}
    data['acc_data'] = acc_data  # Include acc_data in the main dictionary
    return data

def preprocess_data(data):
    """Process all datasets: reset index, add date columns, and handle specific value replacements."""
    processed_data = {}
    for name, df in data.items():
        df = df.reset_index(drop=True)
        
        if 'timestamp' in df.columns:
            df['date'] = df['timestamp'].dt.date
        
        if name == 'hr_data':
            df['heart_rate'].replace(0, np.nan, inplace=True)
        
        processed_data[name] = df
    
    return processed_data

def load_and_preprocess_data(base_path):
    """Load and preprocess all data from the base path."""
    data = load_parquet_data(base_path)
    return preprocess_data(data)

# Set your base path to the location where your files are stored.

# Load and preprocess data
data = load_and_preprocess_data(base_path)
print('Data loading and preprocessing completed.')

# Access specific datasets like this:
acc_data = data['acc_data']
activity_data = data['activity_data']
hr_data = data['hr_data']
step_count = data['step_count']
light = data['light']
gps = data['gps']


100% 4/4 [01:21<00:00, 20.42s/it]


Data loading and preprocessing completed.


In [3]:
acc_data

,subject_id,timestamp,x,y,z,date
0,1,2023-08-20 00:00:00.025,0.933201,-3.522235,9.164511,2023-08-20
1,1,2023-08-20 00:00:00.043,0.947558,-3.522235,9.169296,2023-08-20
2,1,2023-08-20 00:00:00.110,0.966700,-3.479164,9.164511,2023-08-20
3,1,2023-08-20 00:00:00.131,0.947558,-3.522235,9.159725,2023-08-20
4,1,2023-08-20 00:00:00.150,0.918844,-3.531806,9.159725,2023-08-20
...,...,...,...,...,...,...
445635019,4,2023-11-01 23:57:59.889,9.256635,0.253639,-3.208177,2023-11-01
445635020,4,2023-11-01 23:57:59.907,9.245867,0.269193,-3.211766,2023-11-01
445635021,4,2023-11-01 23:57:59.926,9.262916,0.242871,-3.190231,2023-11-01
445635022,4,2023-11-01 23:57:59.947,9.274879,0.264407,-3.223730,2023-11-01


In [4]:
activity_data

,subject_id,timestamp,m_activity,date
0,1,2023-08-20 00:00:12.814,3,2023-08-20
1,1,2023-08-20 00:01:12.814,3,2023-08-20
2,1,2023-08-20 00:02:12.815,3,2023-08-20
3,1,2023-08-20 00:03:12.813,3,2023-08-20
4,1,2023-08-20 00:04:12.813,3,2023-08-20
...,...,...,...,...
149865,4,2023-11-01 23:55:17.408,3,2023-11-01
149866,4,2023-11-01 23:56:17.406,3,2023-11-01
149867,4,2023-11-01 23:57:17.406,3,2023-11-01
149868,4,2023-11-01 23:58:17.406,3,2023-11-01


In [5]:
hr_data

,subject_id,timestamp,heart_rate,date
0,1,2023-08-20 00:00:44.572,NaN,2023-08-20
1,1,2023-08-20 00:01:44.752,NaN,2023-08-20
2,1,2023-08-20 00:02:44.919,NaN,2023-08-20
3,1,2023-08-20 00:03:45.075,NaN,2023-08-20
4,1,2023-08-20 00:04:45.248,NaN,2023-08-20
...,...,...,...,...
130798,4,2023-11-01 23:30:38.868,172.0,2023-11-01
130799,4,2023-11-01 23:34:42.690,154.0,2023-11-01
130800,4,2023-11-01 23:35:54.649,167.0,2023-11-01
130801,4,2023-11-01 23:43:57.358,170.0,2023-11-01


In [6]:
step_count

,subject_id,timestamp,burned_calories,distance,running_steps,speed,steps,step_frequency,walking_steps,date
0,1,2023-08-20 00:00:00,0.000000,0.000000,0,0.000000,0,0.000000,0,2023-08-20
1,1,2023-08-20 09:41:00,5.279053,62.480469,40,8.440000,75,1.250000,35,2023-08-20
2,1,2023-08-20 09:42:00,2.160278,30.285156,0,3.405882,40,0.666667,40,2023-08-20
3,1,2023-08-20 09:43:00,0.719971,19.941406,0,2.828571,27,0.450000,27,2023-08-20
4,1,2023-08-20 09:44:00,2.809692,42.910156,12,4.751613,56,0.933333,44,2023-08-20
...,...,...,...,...,...,...,...,...,...,...
103212,4,2023-11-01 22:30:00,0.000000,0.000000,0,0.000000,0,0.000000,0,2023-11-01
103213,4,2023-11-01 22:31:00,0.000000,0.000000,0,0.000000,0,0.000000,0,2023-11-01
103214,4,2023-11-01 22:32:00,0.000000,0.000000,0,0.000000,0,0.000000,0,2023-11-01
103215,4,2023-11-01 22:33:00,0.000000,0.000000,0,0.000000,0,0.000000,0,2023-11-01


In [7]:
light

,subject_id,timestamp,m_light,date
0,1,2023-08-20 00:02:00,254.0,2023-08-20
1,1,2023-08-20 00:12:00,275.0,2023-08-20
2,1,2023-08-20 00:22:00,261.0,2023-08-20
3,1,2023-08-20 00:32:00,107.0,2023-08-20
4,1,2023-08-20 00:42:00,105.0,2023-08-20
...,...,...,...,...
14901,4,2023-11-01 23:18:00,0.0,2023-11-01
14902,4,2023-11-01 23:28:00,0.0,2023-11-01
14903,4,2023-11-01 23:38:00,0.0,2023-11-01
14904,4,2023-11-01 23:48:00,0.0,2023-11-01


In [8]:
gps

,subject_id,timestamp,altitude,latitude,longitude,speed,date
0,1,2023-08-20 00:00:08,144.217651,0.016095,0.926485,1.437909e-01,2023-08-20
1,1,2023-08-20 00:00:13,144.217651,0.016090,0.926477,1.607707e-01,2023-08-20
2,1,2023-08-20 00:00:18,144.217651,0.016091,0.926478,6.571416e-03,2023-08-20
3,1,2023-08-20 00:00:23,144.217651,0.016091,0.926474,5.931024e-02,2023-08-20
4,1,2023-08-20 00:00:28,144.217651,0.016092,0.926477,4.945369e-02,2023-08-20
...,...,...,...,...,...,...,...
955234,4,2023-11-01 23:59:37,198.432245,0.465537,0.395172,1.437337e-15,2023-11-01
955235,4,2023-11-01 23:59:42,198.475365,0.465537,0.395172,1.437337e-15,2023-11-01
955236,4,2023-11-01 23:59:47,198.418384,0.465537,0.395172,1.437337e-15,2023-11-01
955237,4,2023-11-01 23:59:52,198.518999,0.465537,0.395172,1.437337e-15,2023-11-01


# parquet 저장 (user, date 별)

In [9]:
def save_grouped_data(df, file_prefix):
    for (subject_id, date), group in df.groupby(['subject_id', 'date']):
        directory = f'datasets/raw_datasets_all_sensor/valid_datasets/{file_prefix}'
        if not os.path.exists(directory):
            os.makedirs(directory)
        file_path = f'{directory}/user{subject_id}_{date}.parquet'
        group.to_parquet(file_path)

acc_data = data['acc_data']
activity_data = data['activity_data']
hr_data = data['hr_data']
step_count = data['step_count']
light = data['light']
gps = data['gps']
activity_data['m_activity'] = activity_data['m_activity'].fillna(4).astype(int)

save_grouped_data(data['acc_data'], 'acc')
save_grouped_data(data['activity_data'], 'activity')
save_grouped_data(data['hr_data'], 'hr')
save_grouped_data(data['step_count'], 'step')
save_grouped_data(data['light'], 'light')
save_grouped_data(data['gps'], 'gps')

# 2. 데이터 병합

In [10]:
import pandas as pd
from datetime import datetime, timedelta

def make_timestamps_unique(df, timestamp_col='timestamp'):
    if not df.empty:
        df = df.sort_values(by=[timestamp_col])
        df[timestamp_col] = df[timestamp_col] + pd.to_timedelta(df.groupby(timestamp_col).cumcount(), unit='ns')
    return df

def merge_data_for_group(user, date):
    data_sources = {}
    sensor_files = ['acc', 'activity', 'hr', 'step', 'light', 'gps']
    
    for sensor in sensor_files:
        file_path = f'datasets/raw_datasets_all_sensor/valid_datasets/{sensor}/user{user}_{date}.parquet'
        try:
            df = pd.read_parquet(file_path)
            if sensor == 'gps':
                df = df.drop('speed', axis=1)
                
            if sensor == 'activity':
                df['m_activity'] = df['m_activity'].astype(int)
            # 삭제할 컬럼이 없는 경우를 처리하기 위해 errors='ignore' 추가
            df = df.drop(columns=['subject_id', 'date'], errors='ignore')
            data_sources[sensor] = df
        except FileNotFoundError:
            columns_map = {
                'acc': ['timestamp', 'x', 'y', 'z'],
                'activity': ['timestamp', 'm_activity'],
                'hr': ['timestamp', 'heart_rate'],
                'step': ['timestamp', 'steps'],
                'light': ['timestamp', 'm_light'],
                'gps': ['timestamp', 'altitude', 'latitude', 'longitude']
            }
            data_sources[sensor] = pd.DataFrame(columns=columns_map[sensor])
        
        if not data_sources[sensor].empty:
            data_sources[sensor]['timestamp'] = pd.to_datetime(data_sources[sensor]['timestamp'])
            data_sources[sensor] = make_timestamps_unique(data_sources[sensor])
            data_sources[sensor] = data_sources[sensor].set_index('timestamp').resample('S').nearest()
        else:
            data_sources[sensor] = pd.DataFrame(index=pd.date_range(start=datetime.strptime(date, '%Y-%m-%d'), periods=86400, freq='S'))

    merged_data = pd.DataFrame(index=pd.date_range(start=datetime.strptime(date, '%Y-%m-%d'), periods=86400, freq='S'))
    for key, df in data_sources.items():
        merged_data = merged_data.join(df, how='left', rsuffix='_duplicate')  # 중복 컬럼명에 접미사 '_duplicate' 추가

    # 필요한 경우, 중복 컬럼 삭제
    duplicate_cols = [col for col in merged_data.columns if '_duplicate' in col]
    merged_data = merged_data.drop(columns=duplicate_cols)
    y_ranges = {}
    for column in merged_data.columns:
        if not merged_data[column].dropna().empty:  # 신뢰구간 계산 전 NaN 제거
            y_ranges[column] = (
                merged_data[column].quantile(0.0001, interpolation='linear'),
                merged_data[column].quantile(0.9999, interpolation='linear')
            )

    # 데이터 보간
    merged_data = merged_data.interpolate(method='linear')

    return merged_data, y_ranges



# 3 시각화

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

def plot_time_series(data, user, date, y_ranges):
    # x축을 00:00:00부터 23:59:59까지 고정
    total_seconds = 86400
    time_range = pd.date_range(start=datetime.strptime(date, '%Y-%m-%d'), periods=total_seconds, freq='S')

    # 데이터를 시간 단위로 정렬
    data = data.reindex(time_range)

    # 시계열 이미지 생성
    fig, axes = plt.subplots(11, 1, figsize=(10, 10), sharex=True, facecolor='black')
    fig.patch.set_facecolor('black')

    for ax in axes:
        ax.set_facecolor('black')
        ax.spines['top'].set_visible(False)   # Hide the top spine
        ax.spines['right'].set_visible(False) # Hide the right spine
        ax.spines['left'].set_visible(False)  # Hide the left spine
        ax.spines['bottom'].set_visible(False)# Hide the bottom spine

    # 설정한 시간 범위에 맞게 x축 설정
    for ax in axes:
        ax.set_xlim([time_range[0], time_range[-1]])

    # 컬럼 인덱스 딕셔너리 생성
    column_indices = {
        'x': 0, 'y': 1, 'z': 2, 'heart_rate': 3, 'm_activity': 4,
        'speed': 5, 'steps': 6,'m_light': 7, 'altitude': 8, 'latitude': 9, 'longitude': 10
    }

    # 각 컬럼에 대한 그래프 플로팅 및 y축 범위 설정
    for column, idx in column_indices.items():
        if column in data.columns:
            axes[idx].plot(data.index, data[column], color='white')
#             axes[idx].set_ylabel(column, color='white')  # 레이블 설정
            # y축 범위 설정
            if column in y_ranges:
                if column == 'm_activity':
                    axes[idx].set_ylim((0, 2))
                else:
                    axes[idx].set_ylim(y_ranges[column])

    plt.tight_layout()  # Make the layout tight
    plt.savefig(f'datasets/image_datasets_all_sensor/user{user}_{date}_valid_all_sensor.png')


# 3. 데이터셋 생성 (Valid)

## 3.1 Valid data 

In [12]:
val_df = pd.read_csv('datasets/val_label.csv')
val_df

,subject_id,date,Q1,Q2,Q3,S1,S2,S3,S4
0,1,2023-08-20,1,1,1,0,0,0,0
1,1,2023-08-21,1,1,1,0,0,1,0
2,1,2023-08-22,0,1,1,0,1,1,0
3,1,2023-08-23,0,1,1,0,0,1,0
4,1,2023-08-24,1,1,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...
100,4,2023-10-27,0,1,0,0,1,1,1
101,4,2023-10-28,1,1,0,1,1,1,1
102,4,2023-10-29,1,1,0,0,1,1,1
103,4,2023-10-30,0,1,0,0,0,1,1


In [ ]:
from tqdm.auto import tqdm

bar = tqdm(range(val_df.shape[0]))

for idx in bar:
    user, date, *rest = val_df.iloc[idx].values
    bar.set_description(f'user: {user}, date: {date}')
    merged_data, y_ranges = merge_data_for_group(user, date)
    plot_time_series(merged_data, user, date, y_ranges)